In [40]:
%load_ext autoreload
%autoreload 2


import sys

sys.path.append("..")

import pandas as pd
import yfinance as yf

from src import metrics

df = yf.download("SPY, TSLA, NVDA, TLT, KO", start="2011-01-01", auto_adjust=True)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[*********************100%***********************]  5 of 5 completed


In [41]:
close = df["Close"]
ret = metrics.daily_returns(prices=close)
vol = metrics.rolling_vol(returns=ret, window=20)

inv_vol = 1 / vol
weights = inv_vol.div(inv_vol.sum(axis=1), axis=0)

weights = weights.shift(1).dropna()
ret_a = ret.loc[weights.index] 
n = weights.shape[1]

eq_weights = pd.DataFrame(1 / n, index=weights.index, columns=weights.columns)

port_iv = (weights * ret_a).sum(axis=1)
port_ew = (eq_weights * ret_a).sum(axis=1)

bt = pd.DataFrame({"inv_vol": port_iv, "equal": port_ew}).dropna()

print(bt.head())
print(bt.index[0])          # should be ~21 trading days after your start
print((bt == 0).sum())      # should be 0 or near it

             inv_vol     equal
Date                          
2011-02-02  0.000447  0.008383
2011-02-03 -0.004538 -0.008479
2011-02-04 -0.001474  0.001608
2011-02-07 -0.000062 -0.009503
2011-02-08  0.003999  0.006567
2011-02-02 00:00:00
inv_vol    0
equal      0
dtype: int64
